# Vanilla GCN on Elliptic++ Transactions

This notebook runs locally in Jupyter on the Elliptic++ Transactions dataset.

Expected files in `data/raw/`:

- `txs_features.csv` — transaction IDs plus 183 node features
- `txs_classes.csv` — transaction IDs and classes (`1` illicit, `2` licit, `unknown` unlabeled)
- `txs_edgelist.csv` — transaction-to-transaction money-flow edges

The model remains the same vanilla GCN:

$$H^{(k)} = \sigma\left(\tilde{A}H^{(k-1)}W^{(k)}\right)$$

The graph is kept sparse so the full 203,769-node dataset can be loaded without constructing a dense adjacency matrix.

## 1. Local setup

In [ ]:
# Install project requirements into the active Jupyter kernel when needed.
import importlib.util
import subprocess
import sys
from pathlib import Path

requirements_path = Path.cwd() / 'requirements.txt'
if not requirements_path.exists():
    requirements_path = Path.cwd().parent / 'requirements.txt'
if requirements_path.exists():
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)],
        check=True,
    )
    print(f'Requirements loaded from {requirements_path}')
else:
    print('requirements.txt not found; continuing with the current environment.')

In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Data directory:', DATA_DIR)
print('PyTorch:', torch.__version__)
print('Device: CPU (full-batch sparse GCN)')

## 2. Sparse graph preprocessing

The transaction graph is normalized as $\tilde{A}=\hat{D}^{-1/2}(A+I)\hat{D}^{-1/2}$, but stored as a PyTorch COO sparse tensor.

In [ ]:
def make_sparse_normalized_adjacency(src, dst, num_nodes):
    """Build sparse A_tilde = D^(-1/2) (A + I) D^(-1/2)."""
    self_nodes = np.arange(num_nodes, dtype=np.int64)
    all_src = np.concatenate([src, dst, self_nodes])
    all_dst = np.concatenate([dst, src, self_nodes])
    values = np.ones(all_src.shape[0], dtype=np.float32)

    indices = torch.tensor(np.vstack([all_src, all_dst]), dtype=torch.long)
    values = torch.tensor(values, dtype=torch.float32)
    adjacency = torch.sparse_coo_tensor(
        indices, values, size=(num_nodes, num_nodes)
    ).coalesce()

    degree = torch.sparse.sum(adjacency, dim=1).to_dense()
    inv_sqrt_degree = degree.clamp_min(1e-12).pow(-0.5)
    normalized_values = (
        adjacency.values()
        * inv_sqrt_degree[adjacency.indices()[0]]
        * inv_sqrt_degree[adjacency.indices()[1]]
    )
    return torch.sparse_coo_tensor(
        adjacency.indices(), normalized_values, adjacency.shape
    ).coalesce()


def make_masks(labels, train_ratio=0.6, val_ratio=0.2, seed=SEED):
    """Split labelled nodes only; unknown nodes stay out of every mask."""
    labelled = np.flatnonzero(labels >= 0)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(labelled)
    n_train = int(len(perm) * train_ratio)
    n_val = int(len(perm) * val_ratio)

    train_mask = np.zeros(len(labels), dtype=bool)
    val_mask = np.zeros(len(labels), dtype=bool)
    test_mask = np.zeros(len(labels), dtype=bool)
    train_mask[perm[:n_train]] = True
    val_mask[perm[n_train:n_train + n_val]] = True
    test_mask[perm[n_train + n_val:]] = True
    return train_mask, val_mask, test_mask

print('Sparse preprocessing functions defined.')

## 3. Load Elliptic++ Transactions

Place the three downloaded CSV files in `data/raw/`, then run the next cell.

The loader reads the transaction IDs, 183 features, labels, and money-flow edges directly from the Elliptic++ CSV files.

In [ ]:
FEATURES_PATH = DATA_DIR / 'txs_features.csv'
CLASSES_PATH = DATA_DIR / 'txs_classes.csv'
EDGES_PATH = DATA_DIR / 'txs_edgelist.csv'

missing = [path for path in (FEATURES_PATH, CLASSES_PATH, EDGES_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing Elliptic++ files:\n' + '\n'.join(f'  - {path}' for path in missing)
    )

print('Reading transaction features...')
features_df = pd.read_csv(FEATURES_PATH, header=None)
tx_ids = features_df.iloc[:, 0].astype(np.int64).to_numpy()
X_np = features_df.iloc[:, 1:].to_numpy(dtype=np.float32)
tx_id_to_idx = {int(tx_id): index for index, tx_id in enumerate(tx_ids)}
num_nodes = len(tx_ids)

print('Reading transaction classes...')
classes_df = pd.read_csv(CLASSES_PATH)
classes_df.columns = classes_df.columns.str.strip()
raw_classes = classes_df.set_index('txId')['class'].astype(str).str.strip()
y_np = np.full(num_nodes, -1, dtype=np.int64)
for tx_id, class_value in raw_classes.items():
    index = tx_id_to_idx.get(int(tx_id))
    if index is not None and class_value == '1':
        y_np[index] = 0
    elif index is not None and class_value == '2':
        y_np[index] = 1

print('Reading transaction edges...')
edges_df = pd.read_csv(EDGES_PATH)
edges_df.columns = edges_df.columns.str.strip()
src_ids = edges_df.iloc[:, 0].astype(np.int64).to_numpy()
dst_ids = edges_df.iloc[:, 1].astype(np.int64).to_numpy()
valid = np.array(
    [(src in tx_id_to_idx and dst in tx_id_to_idx) for src, dst in zip(src_ids, dst_ids)],
    dtype=bool,
)
src = np.array([tx_id_to_idx[value] for value in src_ids[valid]], dtype=np.int64)
dst = np.array([tx_id_to_idx[value] for value in dst_ids[valid]], dtype=np.int64)

train_mask, val_mask, test_mask = make_masks(y_np)
y_safe = y_np.copy()
y_safe[y_safe < 0] = 0

A_tilde = make_sparse_normalized_adjacency(src, dst, num_nodes)
X_pt = torch.from_numpy(X_np)
y_pt = torch.from_numpy(y_safe)
train_mask_pt = torch.from_numpy(train_mask)
val_mask_pt = torch.from_numpy(val_mask)
test_mask_pt = torch.from_numpy(test_mask)

print(f'Nodes: {num_nodes:,}')
print(f'Features: {X_np.shape[1]}')
print(f'Unique normalized edges including self-loops: {A_tilde._nnz():,}')
print(f'Labelled: {(y_np >= 0).sum():,} | illicit: {(y_np == 0).sum():,} | licit: {(y_np == 1).sum():,}')
print(f'Train/val/test: {train_mask.sum():,}/{val_mask.sum():,}/{test_mask.sum():,}')

## 4. Train the vanilla GCN

These values match `configs/default.yaml`. The architecture is unchanged: two GCN layers with a 64-dimensional hidden representation.

In [ ]:
HIDDEN_DIM = 64
NUM_LAYERS = 2
EPOCHS = 200
LR = 0.001
WEIGHT_DECAY = 5e-4
LOG_EVERY = 10
NUM_CLASSES = 2
INPUT_DIM = X_np.shape[1]

print(f'Input features: {INPUT_DIM}')
print(f'Architecture: GCN({INPUT_DIM} -> {HIDDEN_DIM} -> {NUM_CLASSES})')
print(f'Training: {EPOCHS} epochs, lr={LR}, weight_decay={WEIGHT_DECAY}')

The graph convolution remains $\tilde{A}H W$; only the adjacency storage is sparse so the full dataset fits in memory.

In [ ]:
class GCNLayer(nn.Module):
    """Vanilla GCN layer: H_out = ReLU(A_tilde @ H @ W)."""

    def __init__(self, input_dim, output_dim, activation=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, output_dim))
        nn.init.xavier_uniform_(self.weight)
        self.activation = activation

    def forward(self, H, A_tilde):
        support = torch.sparse.mm(A_tilde, H)
        output = support @ self.weight
        return F.relu(output) if self.activation else output


class VanillaGCN(nn.Module):
    """Two-layer vanilla GCN with the same architecture as the project."""

    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2):
        super().__init__()
        if num_layers != 2:
            raise ValueError('This notebook uses the configured two-layer architecture.')
        self.layers = nn.ModuleList([
            GCNLayer(input_dim, hidden_dim, activation=True),
            GCNLayer(hidden_dim, num_classes, activation=False),
        ])

    def forward(self, X, A_tilde, return_hidden=False):
        hidden = self.layers[0](X, A_tilde)
        logits = self.layers[1](hidden, A_tilde)
        return (logits, hidden) if return_hidden else logits


model = VanillaGCN(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES, NUM_LAYERS)
print(f'Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def accuracy(logits, labels, mask):
    predictions = logits[mask].argmax(dim=1)
    return (predictions == labels[mask]).float().mean().item() * 100

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(X_pt, A_tilde)
    train_loss = criterion(logits[train_mask_pt], y_pt[train_mask_pt])
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        eval_logits = model(X_pt, A_tilde)
        val_loss = criterion(eval_logits[val_mask_pt], y_pt[val_mask_pt]).item()
        train_acc = accuracy(eval_logits, y_pt, train_mask_pt)
        val_acc = accuracy(eval_logits, y_pt, val_mask_pt)

    history['train_loss'].append(train_loss.item())
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if epoch == 1 or epoch % LOG_EVERY == 0:
        print(
            f'Epoch {epoch:3d}/{EPOCHS} | '
            f'train loss {train_loss.item():.4f} | val loss {val_loss:.4f} | '
            f'train acc {train_acc:.2f}% | val acc {val_acc:.2f}%'
        )

model.eval()
with torch.no_grad():
    final_logits, final_hidden = model(X_pt, A_tilde, return_hidden=True)
    predictions = final_logits.argmax(dim=1)
    test_accuracy = accuracy(final_logits, y_pt, test_mask_pt)

print(f'Finished in {time.time() - start_time:.1f}s')
print(f'Test accuracy: {test_accuracy:.2f}%')

---
## Section 6 — 📈 Compare & Visualise

In [ ]:
# Save the trained model, predictions, embeddings, and training curves locally.
checkpoint_path = OUTPUT_DIR / 'gcn_pytorch_checkpoint.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'input_dim': INPUT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_classes': NUM_CLASSES,
        'num_layers': NUM_LAYERS,
    },
    'test_accuracy': test_accuracy,
}, checkpoint_path)

np.save(OUTPUT_DIR / 'node_embeddings.npy', final_hidden.detach().numpy())
p.save(OUTPUT_DIR / 'predictions.npy', predictions.detach().numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Percent')
axes[1].legend()
axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

print('Saved outputs to:', OUTPUT_DIR)
print('Checkpoint:', checkpoint_path.name)
print(f'Final test accuracy: {test_accuracy:.2f}%')

## 5. Outputs

The final cell writes the checkpoint, predictions, node embeddings, and training curves to `outputs/notebook/`.